# PG-Stack · Step 4c — PG-Boost v2: stronger graph-outlier features + rigorous testing

Step 4b: graph-outlier distances were the **top-2 features**, but *local adds to global* was only borderline (p = 0.097).
This notebook strengthens the claim:

1. **Less noise** — 30 LightGBM bags per fold for the main models (was 5), 1,000 patient-bootstrap resamples, **Holm correction**.
2. **Richer graph-outlier features** on **3 graphs** (appearance / DINOv2, anatomic / 3D body, tabular feature space), **multi-scale k = 5, 10, 20**: mean kNN distance + proper **LOF**; plus per-patient **Mahalanobis** distance and **Isolation-Forest** score; body-site-adjusted density.
3. **Honest attribution** — group-drop ablations instead of split-count importance.
4. **Confound check** — is `knn_pos_dist` just body-region crowding?

CPU is enough. Runtime ≈ 60–80 min. Reads/writes `/kaggle/working`.

In [ ]:
import os, sys, glob, time, gc, warnings
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
COMP_DIR = "/kaggle/input/competitions/isic-2024-challenge"
SEED, NEG_SAMPLE_RATIO = 42, 0.01
N_BAGS_MAIN, N_BAGS_ABL = 30, 15      # main models vs ablations
N_BOOT = 1000
KS = [5, 10, 20]
K_DIFF = 10                           # neighbourhood for local feature differences (as in 4b)

def locate(name):
    # look in /kaggle/working first, then in attached inputs (fixed depth: never scans the image folder)
    if os.path.exists(f"{WORK}/{name}"): return f"{WORK}/{name}"
    for d in range(1, 5):
        hits = glob.glob("/kaggle/input/" + "*/" * d + name)
        if hits: return hits[0]
    return None

NEED = ["oof_step3.csv", "emb_pca32.npy", "pgstack_eval.py"]
paths = {n: locate(n) for n in NEED}
missing = [n for n, p in paths.items() if p is None]
if missing:
    raise FileNotFoundError(
        f"Missing {missing}. The Kaggle session was probably reset (files in /kaggle/working are deleted). "
        "Fix: upload your backed-up files (Add Input -> Upload -> New Dataset), or re-run Steps 1-3.")
sys.path.insert(0, os.path.dirname(paths["pgstack_eval.py"]))
from pgstack_eval import *
assert os.path.exists(f"{COMP_DIR}/train-metadata.csv"), "check COMP_DIR"
print({n: p for n, p in paths.items()})

## 1. Lesion features (identical to Steps 1 / 4b)

In [ ]:
oof = pd.read_csv(paths["oof_step3.csv"])
df = pd.read_csv(f"{COMP_DIR}/train-metadata.csv", low_memory=False)
df = oof[["isic_id", "fold"]].merge(df, on="isic_id", how="left")
assert (df.isic_id.values == oof.isic_id.values).all()
emb = np.load(paths["emb_pca32.npy"]).astype(np.float32)
assert len(emb) == len(oof)

LEAK = ["lesion_id", "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
        "mel_mitotic_index", "mel_thick_mm", "tbp_lv_dnn_lesion_confidence"]
NUM_RAW = [c for c in df.columns if (c.startswith("tbp_lv_") or c in ["age_approx", "clin_size_long_diam_mm"])
           and c not in LEAK and pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = ["sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location", "tbp_lv_location_simple"]

def add_lesion_features(d):
    d = d.copy(); eps = 1e-6
    d["age_approx"] = pd.to_numeric(d["age_approx"], errors="coerce")
    d["lesion_size_ratio"] = d.tbp_lv_minorAxisMM / (d.clin_size_long_diam_mm + eps)
    d["lesion_shape_index"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM ** 2 + eps)
    d["hue_contrast"] = (d.tbp_lv_H - d.tbp_lv_Hext).abs()
    d["luminance_contrast"] = (d.tbp_lv_L - d.tbp_lv_Lext).abs()
    d["lesion_color_difference"] = np.sqrt(d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2)
    d["border_complexity"] = d.tbp_lv_norm_border + d.tbp_lv_symm_2axis
    d["color_uniformity"] = d.tbp_lv_color_std_mean / (d.tbp_lv_radial_color_std_max + eps)
    d["position_distance_3d"] = np.sqrt(d.tbp_lv_x**2 + d.tbp_lv_y**2 + d.tbp_lv_z**2)
    d["perimeter_to_area_ratio"] = d.tbp_lv_perimeterMM / (d.tbp_lv_areaMM2 + eps)
    d["area_to_perimeter_ratio"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM + eps)
    d["lesion_visibility_score"] = d.tbp_lv_deltaLBnorm + d.tbp_lv_norm_color
    d["symmetry_border_consistency"] = d.tbp_lv_symm_2axis * d.tbp_lv_norm_border
    d["color_shape_composite"] = (d.tbp_lv_color_std_mean + d.tbp_lv_area_perim_ratio + d.tbp_lv_symm_2axis) / 3
    d["log_area"] = np.log1p(d.tbp_lv_areaMM2)
    d["normalized_lesion_size"] = d.clin_size_long_diam_mm / (d.age_approx + eps)
    d["mean_hue_difference"] = (d.tbp_lv_H + d.tbp_lv_Hext) / 2
    d["std_dev_contrast"] = np.sqrt((d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2) / 3)
    return d

ENG = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast", "luminance_contrast", "lesion_color_difference",
       "border_complexity", "color_uniformity", "position_distance_3d", "perimeter_to_area_ratio",
       "area_to_perimeter_ratio", "lesion_visibility_score", "symmetry_border_consistency",
       "color_shape_composite", "log_area", "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast"]
df = add_lesion_features(df)
LESION_NUM = sorted(set(NUM_RAW + ENG))
for c in CAT_COLS: df[c] = df[c].fillna("NA").astype("category")
y, folds, pid = oof.target.values, oof.fold.values, oof.patient_id.values
N_FOLDS = int(folds.max()) + 1
POS_COLS = ["tbp_lv_x", "tbp_lv_y", "tbp_lv_z", "position_distance_3d"]
TAB_COLS = [c for c in LESION_NUM if c not in POS_COLS]        # appearance/shape only, for the tabular graph
print(len(df), "lesions |", len(LESION_NUM), "lesion features")

## 2. Global patient context (as Step 1)

In [ ]:
g = df.groupby("patient_id"); GLOBAL = []
for c in LESION_NUM:
    df[f"{c}_pnorm"] = (df[c] - g[c].transform("mean")) / (g[c].transform("std") + 1e-6); GLOBAL.append(f"{c}_pnorm")
df["patient_n_lesions"] = g["isic_id"].transform("size"); GLOBAL.append("patient_n_lesions")

## 3. Graph features on three graphs
- **app** = DINOv2 embedding (looks similar) · **pos** = 3D body position (nearby) · **tab** = standardised shape/colour features
- **Local differences** (k = 10, as 4b): `(x − mean_neighbours) / std_neighbours` for app and pos graphs
- **Outlier scores** at k = 5, 10, 20 on each graph: mean kNN distance, distance ratio to patient median, proper **LOF**
- Per patient: **Mahalanobis** distance (10 PCs) and **Isolation-Forest** score; site-adjusted body density

In [ ]:
X = df[LESION_NUM].astype(np.float32); X = X.fillna(X.median()).values
XYZ = df[["tbp_lv_x", "tbp_lv_y", "tbp_lv_z"]].fillna(0).values.astype(np.float32)
tab_idx = [LESION_NUM.index(c) for c in TAB_COLS]
TAB = StandardScaler().fit_transform(X[:, tab_idx]).clip(-6, 6).astype(np.float32)   # unsupervised scaling
TAB10 = PCA(10, random_state=SEED).fit_transform(TAB).astype(np.float32)
SPACES = {"app": emb, "pos": XYZ, "tab": TAB}

n, F, KMAX = len(df), X.shape[1], max(KS)
ldiff = {s: np.zeros((n, F), np.float32) for s in ["app", "pos"]}
OUT = {}
for s in SPACES:
    for k in KS:
        for kind in ["dist", "drel", "lof"]:
            OUT[f"g_{s}_k{k}_{kind}"] = np.full(n, np.nan, np.float32)
OUT["g_tab_mahal"] = np.full(n, np.nan, np.float32)
OUT["g_iforest"] = np.full(n, np.nan, np.float32)

def lof_from_knn(dist, idx, k):
    d, i = dist[:, :k], idx[:, :k]
    kdist = d[:, -1]
    reach = np.maximum(d, kdist[i])
    lrd = 1.0 / (reach.mean(1) + 1e-6)
    return lrd[i].mean(1) / lrd

t0 = time.time()
for j, (p, rows) in enumerate(pd.Series(np.arange(n)).groupby(pid).indices.items()):
    r = np.asarray(rows); m = len(r)
    if m < 6: continue
    kmax = min(KMAX, m - 1)
    for s, Z in SPACES.items():
        dist, idx = NearestNeighbors(n_neighbors=kmax + 1).fit(Z[r]).kneighbors(Z[r])
        dist, idx = dist[:, 1:], idx[:, 1:]
        for k in KS:
            kk = min(k, kmax)
            md_ = dist[:, :kk].mean(1)
            OUT[f"g_{s}_k{k}_dist"][r] = md_
            OUT[f"g_{s}_k{k}_drel"][r] = md_ / (np.median(md_) + 1e-6)
            OUT[f"g_{s}_k{k}_lof"][r] = lof_from_knn(dist, idx, kk)
        if s in ldiff:
            kk = min(K_DIFF, kmax); nb = X[r][idx[:, :kk]]
            ldiff[s][r] = (X[r] - nb.mean(1)) / (nb.std(1) + 1e-3)
    if m >= 15:                                        # per-patient Mahalanobis in tabular PC space
        Zp = TAB10[r]; mu = Zp.mean(0); C = np.cov(Zp, rowvar=False) + 1e-3 * np.eye(Zp.shape[1])
        dz = Zp - mu; OUT["g_tab_mahal"][r] = np.sqrt(np.einsum("ij,jk,ik->i", dz, np.linalg.inv(C), dz))
    if m >= 10:                                        # per-patient isolation forest (appearance + shape)
        Zi = np.c_[TAB10[r], emb[r, :10]]
        OUT["g_iforest"][r] = -IsolationForest(n_estimators=100, random_state=SEED).fit(Zi).score_samples(Zi)
    if j % 200 == 0: print(f"  patient {j} | {time.time()-t0:.0f}s")
print(f"graph features built in {time.time()-t0:.0f}s")

for k, v in OUT.items(): df[k] = v
# body-site-adjusted density: is the lesion isolated *relative to its own body region*?
key = df.patient_id.astype(str) + "|" + df.anatom_site_general.astype(str)
df["g_pos_k10_dist_siteadj"] = df["g_pos_k10_dist"] / (df.groupby(key)["g_pos_k10_dist"].transform("median") + 1e-6)

LDIFF_APP = []; LDIFF_POS = []
for jj, c in enumerate(LESION_NUM):
    df[f"{c}_lapp"] = ldiff["app"][:, jj]; LDIFF_APP.append(f"{c}_lapp")
    df[f"{c}_lpos"] = ldiff["pos"][:, jj]; LDIFF_POS.append(f"{c}_lpos")
del ldiff; gc.collect()

OUT_V1 = ["g_app_k10_dist", "g_app_k10_drel", "g_pos_k10_dist", "g_pos_k10_drel"]      # ~ the 4 scores of Step 4b
OUT_ALL = list(OUT.keys()) + ["g_pos_k10_dist_siteadj"]
BASE_F = LESION_NUM + CAT_COLS
print("global", len(GLOBAL), "| local diffs", len(LDIFF_APP) + len(LDIFF_POS), "| outlier scores", len(OUT_ALL))

## 4. Confound check — is `knn_pos_dist` only body-region crowding?

In [ ]:
def auc_within(col, by):
    out = []
    for _, d in df.groupby(by):
        if d.target.nunique() == 2 and d[col].notna().all(): out.append((roc_auc_score(d.target, d[col]), d.target.sum()))
    a = np.array(out); return float((a[:, 0] * a[:, 1]).sum() / a[:, 1].sum())   # positive-weighted mean

rows = []
for col in ["g_pos_k10_dist", "g_pos_k10_dist_siteadj", "g_pos_k10_lof", "g_app_k10_dist", "g_app_k10_lof",
            "g_tab_k10_lof", "g_tab_mahal", "g_iforest"]:
    v = df[col].fillna(df[col].median())
    rows.append(dict(feature=col, auroc_overall=roc_auc_score(y, v),
                     auroc_within_site=auc_within(col, "anatom_site_general"),
                     auroc_within_patient=auc_within(col, "patient_id")))
conf = pd.DataFrame(rows).round(4)
print(conf.to_string(index=False))
print("\nIf AUROC stays well above 0.5 *within site* and *within patient*, the signal is not just body-region crowding.")

## 5. Feature sets + LightGBM (30 bags for main models, 15 for ablations)

In [ ]:
POS_FEATS = set(LDIFF_POS) | {c for c in OUT_ALL if c.startswith("g_pos")}
G1 = BASE_F + GLOBAL
G3 = BASE_F + GLOBAL + LDIFF_APP + LDIFF_POS + OUT_V1                 # = Step 4b PG-Boost
G4 = BASE_F + GLOBAL + LDIFF_APP + LDIFF_POS + OUT_ALL                # = PG-Boost v2
FSETS = {
    "G1_global": (G1, N_BAGS_MAIN),
    "G3_pgboost_v1": (G3, N_BAGS_MAIN),
    "G4_pgboost_v2": (G4, N_BAGS_MAIN),
    "abl_no_localdiff": ([c for c in G4 if c not in set(LDIFF_APP + LDIFF_POS)], N_BAGS_ABL),
    "abl_no_outlier":   ([c for c in G4 if c not in set(OUT_ALL)], N_BAGS_ABL),
    "abl_no_global":    ([c for c in G4 if c not in set(GLOBAL)], N_BAGS_ABL),
    "abl_no_posgraph":  ([c for c in G4 if c not in POS_FEATS], N_BAGS_ABL),
}
LGB_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7, min_child_samples=40,
                  feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=1.0,
                  n_estimators=300, verbosity=-1)

def train_oof(feats, n_bags, name):
    pred_all = np.zeros(n)
    for f in range(N_FOLDS):
        tr = np.where(folds != f)[0]; va = np.where(folds == f)[0]
        pos, negs = tr[y[tr] == 1], tr[y[tr] == 0]
        Xva = df.iloc[va][feats]; pred = np.zeros(len(va))
        for b in range(n_bags):
            rs = SEED + 1000 * f + b; rng = np.random.default_rng(rs)
            sub = np.r_[pos, rng.choice(negs, int(len(negs) * NEG_SAMPLE_RATIO), replace=False)]
            m = lgb.LGBMClassifier(**LGB_PARAMS, random_state=rs).fit(df.iloc[sub][feats], y[sub])
            pred += m.predict_proba(Xva)[:, 1] / n_bags
        pred_all[va] = pred
    return pred_all

prev = locate("oof_step4c.csv")
done = pd.read_csv(prev) if prev else pd.DataFrame({"isic_id": oof.isic_id})
for name, (feats, nb) in FSETS.items():
    col = f"pgb2_{name}"
    if col in done.columns:
        oof[col] = done[col].values; print(f"{name}: loaded from previous run"); continue
    t0 = time.time()
    oof[col] = train_oof(feats, nb, name); done[col] = oof[col].values
    done.to_csv(f"{WORK}/oof_step4c.csv", index=False)        # resumable
    print(f"== {name}: {len(feats)} feats, {nb} bags, pAUC80={pauc_80(y, oof[col]):.4f}  ({(time.time()-t0)/60:.1f} min)")

## 6. Final systems: LR stack with the image learners

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def lr_stack(cols):
    Z = np.stack([logit(oof[c].values) for c in cols], 1); pred = np.zeros(n)
    for f in range(N_FOLDS):
        tr, va = folds != f, folds == f; sc = StandardScaler().fit(Z[tr])
        pred[va] = LogisticRegression(max_iter=2000).fit(sc.transform(Z[tr]), y[tr]).predict_proba(sc.transform(Z[va]))[:, 1]
    return pred
oof["stack_global"] = lr_stack(["pgb2_G1_global", "oof_emb_lr", "oof_cnn"])
oof["stack_v1"]     = lr_stack(["pgb2_G3_pgboost_v1", "oof_emb_lr", "oof_cnn"])
oof["stack_v2"]     = lr_stack(["pgb2_G4_pgboost_v2", "oof_emb_lr", "oof_cnn"])

## 7. Results, 1,000-resample patient bootstrap, Holm-corrected tests

In [ ]:
MODELS = {"tab_pnorm(step1)": "oof_tab_pnorm", "stack_xgb_all": "stack_xgb_all",
          **{k: f"pgb2_{k}" for k in FSETS}, "stack_global": "stack_global",
          "stack_v1": "stack_v1", "stack_v2": "stack_v2"}

t0 = time.time()
boot = PatientBootstrap(pid, n_boot=N_BOOT, seed=SEED)
S = {m: [] for m in MODELS}                      # each model scored ONCE per resample -> reused for all tests
for idx in boot.samples:
    if y[idx].sum() == 0: continue
    for m, c in MODELS.items(): S[m].append(pauc_80(y[idx], oof[c].values[idx]))
S = {m: np.array(v) for m, v in S.items()}
print(f"bootstrap: {time.time()-t0:.0f}s")

rows = []
for m, c in MODELS.items():
    r = full_report(m, y, oof[c].values, pid); mu, sd = fold_stats(y, oof[c].values, folds)
    r.update(pauc80_95CI=f"[{np.percentile(S[m], 2.5):.4f}, {np.percentile(S[m], 97.5):.4f}]", pauc80_fold=f"{mu:.4f} ± {sd:.4f}")
    rows.append(r)
res = pd.DataFrame(rows).set_index("model")
cols = ["pauc80", "pauc80_95CI", "pauc80_fold", "auroc", "sens_spec95", "spec_sens95", "top5_recall", "top10_recall"]
print(res[cols].round(4).to_string())

In [ ]:
def paired(a, b):
    d = S[a] - S[b]
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), (d <= 0).mean()

# pre-registered primary hypotheses (one-sided: new > base) -> Holm correction
PRIMARY = [("G4_pgboost_v2", "G1_global"), ("G3_pgboost_v1", "G1_global"),
           ("stack_v2", "stack_global"), ("stack_v2", "tab_pnorm(step1)"), ("stack_v2", "stack_xgb_all")]
SECONDARY = [("G4_pgboost_v2", "G3_pgboost_v1")] + [(a, "G4_pgboost_v2") for a in FSETS if a.startswith("abl_")]

def table(pairs, holm=False):
    t = []
    for a, b in pairs:
        d, lo, hi, p = paired(a, b)
        t.append(dict(comparison=f"{a} vs {b}", delta_pauc=round(d, 4), ci95=f"[{lo:+.4f}, {hi:+.4f}]", p=p))
    t = pd.DataFrame(t)
    if holm:
        order = np.argsort(t.p.values); adj = np.zeros(len(t)); run = 0
        for rank, i in enumerate(order):
            run = max(run, min(1.0, (len(t) - rank) * t.p.values[i])); adj[i] = run
        t["p_holm"] = adj.round(4); t["significant_0.05"] = t.p_holm < 0.05
    t["p"] = t.p.round(4); return t

prim, sec = table(PRIMARY, holm=True), table(SECONDARY)
print("PRIMARY (Holm-corrected):"); print(prim.to_string(index=False))
print("\nABLATIONS (negative delta = removing that group HURTS = the group is useful):"); print(sec.to_string(index=False))

In [ ]:
keep = ["isic_id", "patient_id", "fold", "target"] + [c for c in oof.columns if c.startswith("pgb2_")] + ["stack_global", "stack_v1", "stack_v2"]
oof[keep].to_csv(f"{WORK}/oof_step4c.csv", index=False)
res[cols].to_csv(f"{WORK}/results_step4c.csv"); prim.to_csv(f"{WORK}/tests_primary_step4c.csv", index=False)
sec.to_csv(f"{WORK}/tests_ablation_step4c.csv", index=False); conf.to_csv(f"{WORK}/confound_step4c.csv", index=False)
print("saved — download oof_step4c.csv + results/tests CSVs as backup")